In [ ]:
!pip install pandas numpy matplotlib seaborn plotly lifelines

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries loaded successfully")

Libraries loaded successfully


# Uploading data
df = pd.read_csv("../data/clinical.project-tcga-brca/clinical.tsv", sep="\t")
print("Shape:", df.shape)
df.head()

In [23]:
columns_of_interest = [
    'case_id',
    'gender',
    'age_at_diagnosis',
    'vital_status',
    'days_to_death',
    'days_to_last_follow_up',
    'ajcc_pathologic_stage',
    'primary_diagnosis',
    'race',
    'ethnicity'
]

# Check columns in file 
available_cols = [col for col in columns_of_interest if col in df.columns]
print("Colonne disponibili:", available_cols)

df_clean = df[available_cols].copy()
df_clean.head()

Colonne disponibili: []


""
0
1
2
3
4


In [24]:
# searching columns by keywords
keywords = ['case_id', 'gender', 'age', 'vital', 'death', 'follow_up', 'stage', 'diagnosis', 'race', 'ethnicity']

for kw in keywords:
    matches = [col for col in df.columns if kw.lower() in col.lower()]
    print(f"'{kw}' -> {matches}")

'case_id' -> ['cases.case_id']
'gender' -> ['demographic.gender']
'age' -> ['demographic.age_at_index', 'demographic.age_is_obfuscated', 'diagnoses.age_at_diagnosis', 'diagnoses.ajcc_clinical_stage', 'diagnoses.ajcc_pathologic_stage', 'diagnoses.ann_arbor_clinical_stage', 'diagnoses.ann_arbor_pathologic_stage', 'diagnoses.cog_liver_stage', 'diagnoses.cog_renal_stage', 'diagnoses.enneking_msts_stage', 'diagnoses.ensat_pathologic_stage', 'diagnoses.esophageal_columnar_dysplasia_degree', 'diagnoses.esophageal_columnar_metaplasia_present', 'diagnoses.figo_stage', 'diagnoses.gastric_esophageal_junction_involvement', 'diagnoses.igcccg_stage', 'diagnoses.inrg_stage', 'diagnoses.inss_stage', 'diagnoses.irs_stage', 'diagnoses.iss_stage', 'diagnoses.masaoka_stage', 'diagnoses.uicc_clinical_stage', 'diagnoses.uicc_pathologic_stage', 'treatments.embolic_agent', 'treatments.radiosensitizing_agent', 'treatments.therapeutic_agents']
'vital' -> ['demographic.vital_status']
'death' -> ['demographic.cau

In [25]:

columns_of_interest = [
    'cases.case_id',
    'demographic.gender',
    'demographic.age_at_index',
    'demographic.vital_status',
    'demographic.days_to_death',
    'diagnoses.days_to_last_follow_up',
    'diagnoses.ajcc_pathologic_stage',
    'diagnoses.primary_diagnosis',
    'demographic.race',
    'demographic.ethnicity'
]

available_cols = [col for col in columns_of_interest if col in df.columns]
print("Colonne disponibili:", available_cols)
print("Numero colonne trovate:", len(available_cols))

df_clean = df[available_cols].copy()
df_clean.head()

Colonne disponibili: ['cases.case_id', 'demographic.gender', 'demographic.age_at_index', 'demographic.vital_status', 'demographic.days_to_death', 'diagnoses.days_to_last_follow_up', 'diagnoses.ajcc_pathologic_stage', 'diagnoses.primary_diagnosis', 'demographic.race', 'demographic.ethnicity']
Numero colonne trovate: 10


,cases.case_id,demographic.gender,demographic.age_at_index,demographic.vital_status,demographic.days_to_death,diagnoses.days_to_last_follow_up,diagnoses.ajcc_pathologic_stage,diagnoses.primary_diagnosis,demographic.race,demographic.ethnicity
0,001cef41-ff86-4d3f-a140-a647ac4b10a1,female,60,Alive,'--,337.0,Stage IA,"Infiltrating duct carcinoma, NOS",white,not hispanic or latino
1,001cef41-ff86-4d3f-a140-a647ac4b10a1,female,60,Alive,'--,337.0,Stage IA,"Infiltrating duct carcinoma, NOS",white,not hispanic or latino
2,001cef41-ff86-4d3f-a140-a647ac4b10a1,female,60,Alive,'--,337.0,Stage IA,"Infiltrating duct carcinoma, NOS",white,not hispanic or latino
3,0045349c-69d9-4306-a403-c9c1fa836644,female,70,Alive,'--,259.0,Stage I,Adenoid cystic carcinoma,white,not hispanic or latino
4,00807dae-9f4a-4fd1-aac2-82eb11bf2afb,female,50,Alive,'--,3102.0,Stage IIB,Apocrine adenocarcinoma,white,not hispanic or latino


In [26]:
print("Total rows:", len(df_clean))
print("unique patient:", df_clean['cases.case_id'].nunique())

Total rows: 5546
unique patient: 1098


In [27]:
print("Total rows:", len(df_clean))
print("unique patient:", df_clean['cases.case_id'].nunique())

# Removing duplicates
df_clean = df_clean.drop_duplicates(subset='cases.case_id', keep='first')

print("\nAfter duplication:")
print("Total rows:", len(df_clean))

Total rows: 5546
unique patient: 1098

After duplication:
Total rows: 1098


In [28]:
print("\nTipi di dato:")
print(df_clean.dtypes)


Tipi di dato:
cases.case_id                       object
demographic.gender                  object
demographic.age_at_index            object
demographic.vital_status            object
demographic.days_to_death           object
diagnoses.days_to_last_follow_up    object
diagnoses.ajcc_pathologic_stage     object
diagnoses.primary_diagnosis         object
demographic.race                    object
demographic.ethnicity               object
dtype: object


In [29]:
# Check unique values in some key columns to detect "missing" placeholders
print(df_clean['demographic.age_at_index'].unique()[:20])
print()
print(df_clean['demographic.days_to_death'].unique()[:20])
print()
print(df_clean['demographic.vital_status'].unique())

['60' '70' '50' '56' '61' '71' '76' '74' '59' '88' '40' '58' '44' '65'
 '55' '64' '62' '34' '57' '49']

["'--" '991' '571' '2534' '1793' '538' '320' '2483' '1812' '255' '3926'
 '723' '1673' '2373' '3941' '2573' '1034' '7455' '584' '2965']

['Alive' 'Dead' "'--"]


In [30]:
import numpy as np

# Replace the "'--" placeholder with a true NaN
df_clean = df_clean.replace("'--", np.nan)


df_clean['demographic.age_at_index'] = pd.to_numeric(df_clean['demographic.age_at_index'], errors='coerce')
df_clean['demographic.days_to_death'] = pd.to_numeric(df_clean['demographic.days_to_death'], errors='coerce')
df_clean['diagnoses.days_to_last_follow_up'] = pd.to_numeric(df_clean['diagnoses.days_to_last_follow_up'], errors='coerce')


print("Missing values after cleaning:")
print(df_clean.isnull().sum())

print("\nNew types:")
print(df_clean.dtypes)

Missing values after cleaning:
cases.case_id                         0
demographic.gender                    1
demographic.age_at_index              1
demographic.vital_status              1
demographic.days_to_death           947
diagnoses.days_to_last_follow_up    107
diagnoses.ajcc_pathologic_stage     102
diagnoses.primary_diagnosis           1
demographic.race                      1
demographic.ethnicity                 1
dtype: int64

New types:
cases.case_id                        object
demographic.gender                   object
demographic.age_at_index            float64
demographic.vital_status             object
demographic.days_to_death           float64
diagnoses.days_to_last_follow_up    float64
diagnoses.ajcc_pathologic_stage      object
diagnoses.primary_diagnosis          object
demographic.race                     object
demographic.ethnicity                object
dtype: object


In [31]:
# Remove the row with almost all missing data
df_clean = df_clean.dropna(subset=['demographic.vital_status'])

df_clean['days_to_event'] = df_clean['demographic.days_to_death'].fillna(
    df_clean['diagnoses.days_to_last_follow_up']
)

print("Row left:", len(df_clean))
print("\nFinal missing values:")
print(df_clean.isnull().sum())

Row left: 1097

Final missing values:
cases.case_id                         0
demographic.gender                    0
demographic.age_at_index              0
demographic.vital_status              0
demographic.days_to_death           946
diagnoses.days_to_last_follow_up    106
diagnoses.ajcc_pathologic_stage     101
diagnoses.primary_diagnosis           0
demographic.race                      0
demographic.ethnicity                 0
days_to_event                        61
dtype: int64


In [32]:
# Deleting patient
df_clean = df_clean.dropna(subset=['days_to_event'])

print("Dataset ready:", df_clean.shape)
df_clean.head()

Dataset ready: (1036, 11)


,cases.case_id,demographic.gender,demographic.age_at_index,demographic.vital_status,demographic.days_to_death,diagnoses.days_to_last_follow_up,diagnoses.ajcc_pathologic_stage,diagnoses.primary_diagnosis,demographic.race,demographic.ethnicity,days_to_event
0,001cef41-ff86-4d3f-a140-a647ac4b10a1,female,60.0,Alive,NaN,337.0,Stage IA,"Infiltrating duct carcinoma, NOS",white,not hispanic or latino,337.0
3,0045349c-69d9-4306-a403-c9c1fa836644,female,70.0,Alive,NaN,259.0,Stage I,Adenoid cystic carcinoma,white,not hispanic or latino,259.0
4,00807dae-9f4a-4fd1-aac2-82eb11bf2afb,female,50.0,Alive,NaN,3102.0,Stage IIB,Apocrine adenocarcinoma,white,not hispanic or latino,3102.0
11,00a2d166-78c9-4687-a195-3d6315c27574,female,56.0,Alive,NaN,5.0,Stage IIA,"Infiltrating duct carcinoma, NOS",white,not hispanic or latino,5.0
31,01263518-5f7c-49dc-8d7e-84b0c03a6a63,female,76.0,Alive,NaN,304.0,Stage IV,"Infiltrating duct carcinoma, NOS",not reported,not reported,304.0


In [33]:
df_clean.to_csv("../data/clinical_clean.csv", index=False)
print("Dataset saved in data/clinical_clean.csv")

Dataset saved in data/clinical_clean.csv
